# Silver Layer Transformation

This notebook transforms Bronze events into a business-ready Silver dataset.

Unlike traditional ETL pipelines that remove problematic records, the Silver layer preserves all original events while enriching them with explicit data quality signals.

The objective is to make data quality observable so downstream business metrics and dashboards can decide how to interpret and use each event.

# Step 1 - Read Bronze Table

Load the Bronze events table from Unity Catalog.

In [0]:
bronze_df = spark.table("workspace.bronze.events_raw")

# Step 2 - Standardize Text Columns

Standardize categorical text columns to ensure consistent grouping and filtering.

This transformation applies defensive normalization only.

No business information is modified.

Applied rules:

- Country → Trim whitespace and convert to uppercase (ISO country code format)
- Platform → Trim whitespace and convert to title case (Web / Mobile)

In [0]:
from pyspark.sql.functions import upper, trim, initcap, col

silver_df = (
    bronze_df
        .withColumn(
            "country",
            upper(trim(col("country")))
        )
        .withColumn(
            "platform",
            initcap(trim(col("platform")))
        )
)

# Step 3 - Detect Duplicate Events

Identify duplicate business events without removing any records.

Business Identity

- event_timestamp
- session_id
- user_id
- event_name

Duplicate Detection Logic

1. Partition events by Business Identity.
2. Order each partition by ingestion_timestamp.
3. The earliest ingested event is treated as the original event.
4. Later ingestions are marked as duplicate events.

This transformation preserves the complete event history while making duplicate events explicitly observable.

New Column

- is_duplicate (Boolean)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, when, col

duplicate_window = Window.partitionBy(
    "event_timestamp",
    "session_id",
    "user_id",
    "event_name"
).orderBy("ingestion_timestamp")

silver_df = (
    silver_df
        .withColumn(
            "row_num",
            row_number().over(duplicate_window)
        )
        .withColumn(
            "is_duplicate",
            when(col("row_num") > 1, True).otherwise(False)
        )
        .drop("row_num")
)

# Validation

In [0]:
display(
    silver_df
        .groupBy("is_duplicate")
        .count()
        .orderBy("is_duplicate")
)

# Step 4 - Calculate Freshness

Calculate event freshness based on the delay between event creation and data ingestion.

New Columns

- freshness_hours
- is_late_arrival

Late Arrival Rule

Events with freshness_hours >= 1 are considered late-arriving events.

The one-hour threshold avoids negligible timestamp precision differences while safely capturing intentionally injected delays (12–48 hours).

In [0]:
from pyspark.sql.functions import (
    col,
    unix_timestamp,
    when
)

In [0]:
silver_df = (
    silver_df
        .withColumn(
            "freshness_hours",
            (
                unix_timestamp(col("ingestion_timestamp"))
                - unix_timestamp(col("event_timestamp"))
            ) / 3600
        )
        .withColumn(
            "is_late_arrival",
            when(col("freshness_hours") >= 1, True).otherwise(False)
        )
)

# Validation 

In [0]:
display(
    silver_df
        .groupBy("is_late_arrival")
        .count()
        .orderBy("is_late_arrival")
)

# Step 5 - Data Quality Summary

Summarize the generated data quality signals.

This step validates the enrichment process before persisting the Silver dataset.

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count

display(
    silver_df.agg(
        count("*").alias("total_events"),
        spark_sum(col("is_duplicate").cast("int")).alias("duplicate_events"),
        spark_sum(col("is_late_arrival").cast("int")).alias("late_arrival_events"),
        avg("freshness_hours").alias("avg_freshness_hours")
    )
)

# Step 6 - Write Silver Table

Persist the enriched Silver dataset into Unity Catalog.
Since the Silver schema evolves from the Bronze schema by introducing additional data quality attributes, overwriteSchema is enabled to update the Delta table metadata during development.

In [0]:
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.events")

# Step 7 - Validate Silver Table

Verify that the Silver table has been successfully created.

In [0]:
silver_validation_df = spark.table(
    "workspace.silver.events"
)

silver_validation_df.printSchema()

display(silver_validation_df.limit(10))

# Step 8 - Transformation Summary

Silver transformation completed successfully.

## New Attributes

- is_duplicate
- freshness_hours
- is_late_arrival

The Silver layer preserves the complete event history while exposing explicit data quality signals for downstream analytics.

The dataset is now ready for Gold layer metric aggregation.